# K40-Adapt: Table 3

This notebook **trains the linear probe and LoRA ranks 1, 2, 4 and 8** at the three paper seeds. Each starts from the supplied pretrained `kryptonet40.pt`, updates only the 70% train rows, and selects the best checkpoint with the 20% validation rows. It evaluates zero-shot inference separately. The fixed 10% local test is scored only after the trainable models are fitted.

In [1]:
NAME="lora"
from pathlib import Path
from datetime import datetime, timezone
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "splits/manifest.json").exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Image
from common import load_split
SEEDS = (42, 123, 2026)
RUN = ROOT / "results/notebook_reruns" / (NAME + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
RUN.mkdir(parents=True)
print("Split seed 67; training seeds", SEEDS)
print("Fresh run directory:", RUN.relative_to(ROOT))

Split seed 67; training seeds (42, 123, 2026)
Fresh run directory: results/notebook_reruns/lora-20261119T021111752707Z


## Train probe and four LoRA ranks on 70%; select checkpoints on 20%

In [2]:
import argparse, json
import torch
from lora.kryptonet import load_kryptonet
from lora.model import count_trainable
from lora.train import build, train as adapt_train, logits
torch.set_num_threads(1)
x_train,y_train,x_val,y_val,_,_=load_split("40-adapt")
tx=torch.from_numpy(np.ascontiguousarray(x_train,dtype=np.float32))
ty=torch.from_numpy(np.ascontiguousarray(y_train,dtype=np.float32))
vx=torch.from_numpy(np.ascontiguousarray(x_val,dtype=np.float32))
vy=torch.from_numpy(np.ascontiguousarray(y_val,dtype=np.float32))
methods=("linear-probe","lora-r1","lora-r2","lora-r4","lora-r8")
models={}; counts={"zero-shot":0}
for method in methods:
    for seed in SEEDS:
        torch.manual_seed(seed)
        rank=int(method.split("r")[-1]) if method.startswith("lora") else 8
        args=argparse.Namespace(rank=rank,alpha=8.0,frozen_head=False)
        base=load_kryptonet(ROOT/"kryptonet40.pt")
        model,_=build("lora" if method.startswith("lora") else method,base,args)
        model,best_epoch,history=adapt_train(model,tx,ty,vx,vy,lr=1e-3 if method.startswith("lora") else 1e-2,max_epochs=300,patience=40,batch_size=128,seed=seed)
        models[method,seed]=model
        counts[method]=count_trainable(model)
        torch.save(model.state_dict(),RUN/f"{method}_seed{seed}.pt")
        (RUN/f"{method}_seed{seed}_history.json").write_text(json.dumps(history))
        print(f'{method} seed {seed}: best validation epoch {best_epoch}, validation accuracy {100*float(np.mean((logits(model,vx).numpy()>0)==y_val)):.2f}%')

linear-probe seed 42: best validation epoch 33, validation accuracy 52.12%


linear-probe seed 123: best validation epoch 55, validation accuracy 52.75%


linear-probe seed 2026: best validation epoch 18, validation accuracy 52.38%


lora-r1 seed 42: best validation epoch 18, validation accuracy 96.25%


lora-r1 seed 123: best validation epoch 22, validation accuracy 96.38%


lora-r1 seed 2026: best validation epoch 26, validation accuracy 96.38%


... (5 outputs left out)

lora-r4 seed 2026: best validation epoch 14, validation accuracy 96.25%


lora-r8 seed 42: best validation epoch 13, validation accuracy 96.38%


lora-r8 seed 123: best validation epoch 13, validation accuracy 96.00%


lora-r8 seed 2026: best validation epoch 14, validation accuracy 96.12%


## Evaluate zero-shot and selected checkpoints on the fixed 10% local test

In [3]:
_,_,_,_,x_test,y_test=load_split("40-adapt")
xt=torch.from_numpy(np.ascontiguousarray(x_test,dtype=np.float32))
rows=[]
for method in ("zero-shot",*methods):
    for seed in SEEDS:
        model=load_kryptonet(ROOT/"kryptonet40.pt") if method=="zero-shot" else models[method,seed]
        predictions=(logits(model,xt).numpy()>0).astype(np.int64)
        rows.append({"method":method,"seed":seed,"correct":int(np.sum(predictions==y_test)),"test_size":len(y_test),"accuracy_pct":100*float(np.mean(predictions==y_test))})
runs=pd.DataFrame(rows)
runs.to_csv(RUN/"local_test_runs.csv",index=False)
summary=runs.groupby("method")["accuracy_pct"].agg(mean="mean",sd="std")
table=[]
for method,label in (("zero-shot","Zero-shot"),("linear-probe","Linear probe"),("lora-r1","LoRA r = 1"),("lora-r2","LoRA r = 2"),("lora-r4","LoRA r = 4"),("lora-r8","LoRA r = 8")):
    result=summary.loc[method]
    table.append({"Method":label,"Trainable parameters":counts[method],"Accuracy (%)":f'{result["mean"]:.2f} ± {result["sd"]:.2f}'})
display(pd.DataFrame(table).set_index("Method"))
print("Saved checkpoints, histories and per-seed results in",RUN.relative_to(ROOT))

,Trainable parameters,Accuracy (%)
Method,,
Zero-shot,0,49.00 ± 0.00
Linear probe,513,49.50 ± 0.43
LoRA r = 1,4137,95.58 ± 0.14
LoRA r = 2,7761,94.75 ± 1.32
LoRA r = 4,15009,95.83 ± 0.29
LoRA r = 8,29505,95.67 ± 0.14


Saved checkpoints, histories and per-seed results in results/notebook_reruns/lora-20261119T021111752707Z
